# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


In [1]:
### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
import pandas as pd

In [4]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [5]:
# TODO: Load environment variables
load_dotenv()

True

In [6]:
assert os.getenv('OPENAI_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

### VectorDB Instance

In [7]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [8]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [9]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

### Add documents

In [10]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [11]:
dict_list = []
for file in os.listdir("games"):
    with open(os.path.join("games",file), "r") as f:
        new_json = json.loads(f.read())
        dict_list.append(new_json)

In [12]:
df = pd.DataFrame(dict_list)
df

,Name,Platform,Genre,Publisher,Description,YearOfRelease
0,Halo Infinite,Xbox Series X|S,First-person shooter,Xbox Game Studios,"The latest installment in the Halo franchise, ...",2021
1,Kinect Adventures!,Xbox 360,Party,Microsoft Game Studios,A collection of mini-games designed to showcas...,2010
2,Minecraft,Xbox One,"Sandbox, Survival",Mojang Studios,A sandbox game that allows players to build an...,2014
3,Super Mario 64,Nintendo 64,Platformer,Nintendo,A groundbreaking 3D platformer that set new st...,1996
4,Pokémon Ruby and Sapphire,Game Boy Advance,Role-playing,Nintendo,Third-generation Pokémon games set in the Hoen...,2002
5,Gran Turismo 5,PlayStation 3,Racing,Sony Computer Entertainment,A comprehensive racing simulator featuring a v...,2010
6,Marvel's Spider-Man 2,PlayStation 5,Action-adventure,Sony Interactive Entertainment,"The sequel to the acclaimed Spider-Man game, f...",2023
7,Wii Sports,Wii,Sports,Nintendo,A collection of sports games that utilize the ...,2006
8,Mario Kart 8 Deluxe,Nintendo Switch,Racing,Nintendo,"An enhanced version of Mario Kart 8, featuring...",2017
9,Pokémon Gold and Silver,Game Boy Color,Role-playing,Nintendo,Second-generation Pokémon games introducing ne...,1999


In [13]:
for coll in chroma_client.list_collections():
    print(coll.name, coll.id, coll.metadata)

udaplay 1d20c7b2-9c73-4034-8496-e28d35c02123 None


In [14]:
all_rows = collection.get()
# all_rows

In [15]:
for metadata in all_rows["metadatas"]:
    print(metadata["Description"], metadata["Genre"], metadata["Name"], metadata["Publisher"], metadata["YearOfRelease"])

A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre. Racing Gran Turismo Sony Computer Entertainment 1997
An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson. Action-adventure Grand Theft Auto: San Andreas Rockstar Games 2004
A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics. Racing Gran Turismo 5 Sony Computer Entertainment 2010
An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains. Action-adventure Marvel's Spider-Man Sony Interactive Entertainment 2018
The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters. Action-adventure Marvel's Spider-Man 2 Sony Interactive Entertainment 2023
Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics. Role-playing Pok

### A Quick Query to check the collection

In [16]:
results = collection.query(
        query_texts=["Grand Turismo"],
        n_results=1
    )
results

{'ids': [['003']],
 'embeddings': None,
 'documents': [['[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.',
    'Platform': 'PlayStation 3',
    'Publisher': 'Sony Computer Entertainment',
    'Genre': 'Racing',
    'YearOfRelease': 2010,
    'Name': 'Gran Turismo 5'}]],
 'distances': [[0.10494053363800049]]}